# SBI812 — Lecture 8
## Statistical Data Analysis with pandas, Seaborn and statsmodels

### Learning outcomes

By the end of this lecture, a student should be able to:

1. distinguish descriptive from inferential statistics;
2. state a research question, null hypothesis and alternative hypothesis;
3. calculate and interpret a confidence interval using software;
4. compare two group means;
5. test treatment differences using one-way ANOVA and interpret Tukey post-hoc comparisons;
6. calculate and visualise correlation;
7. fit and interpret simple and multiple linear-regression models; and
8. report statistical results without confusing statistical significance with biological importance.

Statistical software performs the repetitive arithmetic. The student's responsibility is to choose a suitable analysis, understand its inputs, check its assumptions and interpret its output.

## 1. Continue with the plant-growth dataset

This lecture uses the same experiment as Lecture 7. The main questions are:

1. Is mean biomass different between Bean and Maize plants?
2. Does nitrogen treatment affect biomass?
3. Is gene expression associated with biomass?
4. Does gene expression remain informative after accounting for treatment, species and temperature?

The dataset must first be imported and cleaned in the same documented way. Statistical modelling does not repair poor-quality input data.

## 2. Import the libraries

`statsmodels` is a Python library designed for statistical modelling and detailed statistical inference. If it is not installed, run the following command in a terminal:

```bash
pip install statsmodels
```

The command is an installation instruction, not part of the statistical analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.stats.weightstats import DescrStatsW, ttest_ind
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sns.set_theme(style="whitegrid")


## 3. Import and clean the data

The cleaning operations are kept visible so the statistical results can be reproduced.

In [ ]:
raw_data = pd.read_csv("SBI812_plant_growth_experiment.csv")
data = raw_data.copy()

data["treatment"] = data["treatment"].str.strip().str.title()
data["species"] = data["species"].str.strip().str.title()
data = data.drop_duplicates()

gene_median = data["gene_expression"].median()
chlorophyll_median = data["chlorophyll_index"].median()

data["gene_expression"] = data["gene_expression"].fillna(gene_median)
data["chlorophyll_index"] = data["chlorophyll_index"].fillna(chlorophyll_median)
data = data.dropna(subset=["biomass_g"])

print("Rows available for analysis:", len(data))
print(data.isna().sum())


## 4. Descriptive and inferential statistics

### Descriptive statistics

Descriptive statistics summarise the observations actually present in the dataset. Examples include means, medians, standard deviations, counts and graphs.

### Inferential statistics

Inferential statistics use a sample to draw conclusions about a larger population while acknowledging sampling uncertainty.

In this experiment:

- the **sample** is the collection of measured plants;
- the **population** is the broader set of plants about which the researchers wish to make conclusions.

A large amount of software output cannot compensate for a biased or unrepresentative sample.

## 5. Begin with group descriptions

Before testing a hypothesis, inspect the group sizes, means and standard deviations.

In [ ]:
species_summary = data.groupby("species")["biomass_g"].agg(
    count="count",
    mean="mean",
    standard_deviation="std"
)

species_summary.round(2)


The `agg()` method allows several summaries to be calculated at once.

- `count="count"` counts non-missing biomass values.
- `mean="mean"` calculates the group mean.
- `standard_deviation="std"` calculates the sample standard deviation.

The names on the left become readable output-column names.

## 6. Confidence interval for a mean

A sample mean is one estimate of the population mean. A 95% confidence interval gives a range of values compatible with the data and the statistical assumptions.

It should not be interpreted as saying that 95% of individual plants fall inside the interval.

`DescrStatsW` stores descriptive-statistical calculations for one numerical sample. No weights are supplied here, so each plant contributes equally.

In [ ]:
bean_biomass = data.loc[
    data["species"] == "Bean",
    "biomass_g"
]

bean_statistics = DescrStatsW(bean_biomass)
bean_lower, bean_upper = bean_statistics.tconfint_mean(alpha=0.05)

print("Bean mean biomass:", round(bean_statistics.mean, 2))
print("95% confidence interval:", round(bean_lower, 2), "to", round(bean_upper, 2))


- `.loc[condition, column]` selects Bean rows and then the biomass column.
- `alpha=0.05` corresponds to a 95% confidence interval because $1-0.05=0.95$.
- `tconfint_mean()` uses the t distribution, which accounts for uncertainty in the estimated standard deviation.

## 7. Hypotheses and p-values

A statistical test begins with two competing statements.

For the species comparison:

- **Null hypothesis, $H_0$:** Bean and Maize have the same population mean biomass.
- **Alternative hypothesis, $H_A$:** Their population mean biomasses differ.

The **p-value** measures how incompatible the observed result is with the null hypothesis, assuming the model and test assumptions are appropriate.

A small p-value is evidence against the null hypothesis. It is not:

- the probability that the null hypothesis is true;
- the probability that the result occurred “by chance”; or
- a measure of biological importance.

## 8. Compare two group means

The Welch two-sample t-test allows the groups to have different variances. The two biomass Series are selected separately before the test is run.

In [ ]:
bean_biomass = data.loc[
    data["species"] == "Bean",
    "biomass_g"
]

maize_biomass = data.loc[
    data["species"] == "Maize",
    "biomass_g"
]

t_statistic, p_value, degrees_of_freedom = ttest_ind(
    bean_biomass,
    maize_biomass,
    usevar="unequal"
)

print("t statistic:", round(t_statistic, 3))
print("degrees of freedom:", round(degrees_of_freedom, 2))
print("p-value:", round(p_value, 4))


`usevar="unequal"` requests Welch's version of the test.

The sign of the t statistic depends on the order of the groups. Here, Bean was supplied first and Maize second. The p-value is used to assess evidence against equal means, while the group means show the direction and size of the observed difference.

Always accompany the test with a figure.

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(
    data=data,
    x="species",
    y="biomass_g",
    hue="species",
    palette="Set2",
    legend=False
)
sns.stripplot(
    data=data,
    x="species",
    y="biomass_g",
    color="black",
    alpha=0.55,
    size=4
)
plt.xlabel("Species")
plt.ylabel("Biomass (g)")
plt.title("Biomass observations for Bean and Maize")
plt.tight_layout()
plt.show()


The box plot summarises each group, while `stripplot()` displays the individual observations. Showing both prevents the summary box from hiding the sample size and distribution.

## 9. Compare more than two groups using ANOVA

There are three treatment groups. Repeatedly applying separate t-tests would increase the chance of false-positive conclusions. One-way analysis of variance (ANOVA) tests all treatment means together.

- $H_0$: all three population treatment means are equal.
- $H_A$: at least one treatment mean differs.

ANOVA begins by fitting an ordinary least-squares model.

### Understanding the formula

```python
biomass_g ~ C(treatment)
```

- the variable on the left of `~` is the outcome;
- the variable on the right is the predictor;
- `C(treatment)` tells statsmodels that treatment is categorical;
- the formula does not mean ordinary mathematical equality.

In [ ]:
treatment_model = ols(
    "biomass_g ~ C(treatment)",
    data=data
).fit()

treatment_anova = anova_lm(
    treatment_model,
    typ=2
)

treatment_anova


Important ANOVA-table columns are:

- `sum_sq`: variation attributed to each source;
- `df`: degrees of freedom;
- `F`: the F statistic;
- `PR(>F)`: the p-value.

If the treatment p-value is small, the data provide evidence that not all treatment means are equal. ANOVA alone does not identify every pair that differs; post-hoc comparisons would be required for that more detailed question.

The group means should still be examined.

In [ ]:
treatment_summary = data.groupby("treatment")["biomass_g"].agg(
    count="count",
    mean="mean",
    standard_deviation="std"
)

treatment_summary.round(2)


### 9.1 Post-hoc comparisons with Tukey’s HSD

The ANOVA asks whether all treatment means are equal. If the overall p-value is below 0.05, use Tukey’s honestly significant difference (HSD) test to see which treatment pairs differ. With three groups, there are three pairwise comparisons. Tukey adjusts for comparing all three pairs and controls the chance of making at least one false-positive claim across the set at 5%, under the model assumptions.

The standard Tukey procedure assumes independent observations, approximately normal errors within groups and similar population variances across treatments. This example compares treatment means pooled across plant species, matching the one-way ANOVA.

Use the same rows as the ANOVA. The code drops rows missing either biomass or treatment and then passes the biomass values and corresponding treatment labels to the test.

In [ ]:
anova_p_value = treatment_anova.loc["C(treatment)", "PR(>F)"]
print(f"Overall ANOVA p-value = {anova_p_value:.4g}")

if anova_p_value < 0.05:
    tukey_data = data.dropna(subset=["biomass_g", "treatment"])
    tukey_result = pairwise_tukeyhsd(
        endog=tukey_data["biomass_g"],
        groups=tukey_data["treatment"],
        alpha=0.05
    )
    print(tukey_result.summary())
else:
    print("The overall ANOVA did not reach the 0.05 threshold.")

### 9.2 Read and report the pairwise results

| Output column | How to read it |
|---|---|
| `group1`, `group2` | The two nitrogen treatments being compared |
| `meandiff` | Mean of group 2 minus mean of group 1, in grams |
| `p-adj` | P-value adjusted for testing all treatment pairs |
| `lower`, `upper` | Bounds of the 95% simultaneous confidence interval for the mean difference |
| `reject` | `True` indicates evidence of a difference at the 0.05 level. `False` indicates insufficient evidence to conclude that the means differ. |

For example, a mean difference of **+2 g** means the second treatment averaged 2 g more biomass than the first. An interval entirely above zero supports a higher mean for group 2. An interval entirely below zero supports a lower mean for group 2. The printed table rounds values, so use `reject` to read the decision if a displayed bound appears to be exactly zero.

**Student activity**

1. Write down the overall ANOVA p-value. Does it meet the criterion for running post-hoc comparisons in this workflow?
2. If Tukey’s test runs, list every pair with `reject = True`. For each, identify which treatment had the higher mean, and report the mean difference, confidence interval and adjusted p-value.
3. Choose a pair with `reject = False`, if there is one. Explain what its confidence interval tells you about the differences that are compatible with the data.
4. Write two sentences for a results section. Report the overall ANOVA in the first sentence and interpret one pairwise result in the second, if available.

## 10. Correlation

Correlation measures the direction and strength of a linear relationship between two numerical variables.

Pearson correlation, written as $r$, ranges from −1 to +1:

| Value of $r$ | General interpretation |
|---:|---|
| Near +1 | Strong positive linear relationship |
| Near 0 | Weak linear relationship |
| Near −1 | Strong negative linear relationship |

Correlation does not establish causation. A relationship may be influenced by treatment, species or another variable.

In [ ]:
numerical_columns = data[[
    "temperature_c",
    "gene_expression",
    "biomass_g",
    "chlorophyll_index",
    "leaf_count"
]]

correlation_table = numerical_columns.corr()
correlation_table.round(2)


A heatmap presents the same correlation table using colour. `annot=True` prints the numerical correlations inside the cells.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation_table,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.title("Correlation among numerical plant measurements")
plt.tight_layout()
plt.show()


## 11. Simple linear regression

Simple linear regression models a numerical outcome using one numerical predictor.

The question is:

> How does expected biomass change as gene expression increases?

The model is:

$$\text{biomass}=\beta_0+\beta_1(\text{gene expression})+\text{error}$$

- $\beta_0$ is the intercept;
- $\beta_1$ is the slope;
- the error represents variation not explained by the straight-line model.

statsmodels formula notation expresses this as `biomass_g ~ gene_expression`.

In [ ]:
simple_model = ols(
    "biomass_g ~ gene_expression",
    data=data
).fit()

simple_coefficients = pd.DataFrame({
    "coefficient": simple_model.params,
    "p_value": simple_model.pvalues,
    "confidence_lower": simple_model.conf_int()[0],
    "confidence_upper": simple_model.conf_int()[1]
})

simple_coefficients.round(4)


The coefficient table contains two rows:

- `Intercept`: predicted biomass when gene expression equals zero;
- `gene_expression`: expected change in biomass for a one-unit increase in gene expression.

The slope's confidence interval describes uncertainty in the estimated relationship. If it excludes zero, that agrees with a small p-value for the slope.

The intercept may lie outside the scientifically observed range and should not automatically receive a biological interpretation.

In [ ]:
print("R-squared:", round(simple_model.rsquared, 3))


$R^2$ is the proportion of variation in the observed outcome explained by the fitted model. A larger $R^2$ does not prove that the relationship is causal or that the model is appropriate.

## 12. Display the regression relationship

`regplot()` displays the observations, fitted regression line and confidence band.

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(
    data=data,
    x="gene_expression",
    y="biomass_g",
    scatter_kws={"alpha": 0.7},
    line_kws={"color": "darkred"}
)
plt.xlabel("Relative gene expression")
plt.ylabel("Biomass (g)")
plt.title("Linear relationship between gene expression and biomass")
plt.tight_layout()
plt.show()


## 13. Multiple linear regression

The simple regression does not account for experimental groups. A multiple model can include several predictors:

```python
biomass_g ~ gene_expression + C(treatment) + C(species) + temperature_c
```

The plus signs mean that all listed predictors are included. They do not mean that the original columns are added together.

The gene-expression coefficient now describes its association with biomass **while the listed treatment, species and temperature variables are held constant in the model**.

In [ ]:
multiple_model = ols(
    "biomass_g ~ gene_expression + C(treatment) + C(species) + temperature_c",
    data=data
).fit()

multiple_coefficients = pd.DataFrame({
    "coefficient": multiple_model.params,
    "p_value": multiple_model.pvalues,
    "confidence_lower": multiple_model.conf_int()[0],
    "confidence_upper": multiple_model.conf_int()[1]
})

multiple_coefficients.round(4)


For a categorical predictor, statsmodels chooses one category as a **reference group**. Other category coefficients describe differences from that reference while the other predictors are held constant.

For example, a row such as `C(species)[T.Maize]` compares Maize with the reference species, Bean.

Inspect model fit separately:

In [ ]:
print("Simple-model R-squared:", round(simple_model.rsquared, 3))
print("Multiple-model R-squared:", round(multiple_model.rsquared, 3))
print("Adjusted R-squared:", round(multiple_model.rsquared_adj, 3))


Adjusted $R^2$ penalises the addition of predictors that contribute little. It is more useful than ordinary $R^2$ when comparing models containing different numbers of predictors.

## 14. Check residuals

A **residual** is:

$$\text{observed outcome}-\text{predicted outcome}$$

Residual plots help assess whether a linear model is reasonable.

- Residuals should be scattered around zero without a strong curve.
- A funnel shape may indicate changing variance.
- Extremely distant points may require investigation.

The plot is a diagnostic tool, not a decorative final figure.

In [ ]:
fitted_biomass = multiple_model.fittedvalues
residuals = multiple_model.resid

plt.figure(figsize=(8, 5))
sns.scatterplot(
    x=fitted_biomass,
    y=residuals
)
plt.axhline(
    y=0,
    color="darkred",
    linestyle="--"
)
plt.xlabel("Predicted biomass (g)")
plt.ylabel("Residual")
plt.title("Residuals against predicted biomass")
plt.tight_layout()
plt.show()


## 15. Statistical significance and practical importance

A result can have a small p-value but a small biological effect. Conversely, an important-looking effect can be estimated imprecisely in a small sample.

A responsible interpretation considers:

1. the estimated effect size;
2. the confidence interval;
3. the p-value;
4. the data distribution and possible outliers;
5. study design and sample size; and
6. whether the difference is biologically meaningful.

Avoid writing only “significant” or “not significant.” State the direction and estimated size of the relationship.

## 16. Example reporting structure

A statistical result can be reported using this sequence:

1. identify the groups or variables;
2. report descriptive values;
3. name the statistical method;
4. report the relevant statistic, interval and p-value;
5. interpret the result in the context of the experiment; and
6. mention an important limitation.

Do not claim that an observational association proves a biological mechanism.

## 17. Summary

The statistical workflow is:

```text
Research question
      ↓
Variable types and study design
      ↓
Descriptive statistics and figures
      ↓
Suitable statistical model
      ↓
Effect estimates, intervals and p-values
      ↓
Assumption checks
      ↓
Scientific interpretation and limitations
```

### Practice activities

1. Calculate a 95% confidence interval for mean Maize biomass.
2. Compare Healthy and Stressed plants using a Welch t-test. State both hypotheses first.
3. Fit an ANOVA model for `chlorophyll_index ~ C(treatment)`. Report the F statistic and p-value. If the ANOVA is significant at 0.05, adapt the Tukey code to compare chlorophyll across treatments. For each pair, report the mean difference, 95% simultaneous confidence interval, adjusted p-value and decision. Explain how median imputation of missing chlorophyll values could affect your interpretation.
4. Calculate the correlation between chlorophyll index and biomass.
5. Fit `chlorophyll_index ~ gene_expression` and interpret its slope.
6. Add species to the model and explain how the interpretation changes.
7. Produce a residual plot for the chlorophyll model.
8. Write a short results paragraph containing an effect estimate, confidence interval, p-value and one limitation.